In [2]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
def library_info(BANK_APP_ID,name):
    # The unique identifier for Dashen Bank's app on the Google Play Store
    

    # Step 1: Get app metadata (rating, installs, description...)
    app_info = app(
        BANK_APP_ID,
        lang='en',    # Language: English
        country='et'  # Country: Ethiopia
    )

    print("=" * 50)
    print(f"{name} Bank App Info")
    print("=" * 50)
    print(f"App Title   : {app_info['title']}")
    print(f"Current Score: {app_info['score']}")
    print(f"Total Ratings: {app_info['ratings']:,}")
    print(f"Total Reviews: {app_info['reviews']:,}")
    print(f"Installs     : {app_info['installs']}")

DASHEN_APP_ID = 'com.dashen.dashensuperapp'
BOA_APP_ID='com.boa.boaMobileBanking'
CBE_APP_ID='com.combanketh.mobilebanking'
library_info(DASHEN_APP_ID,'Dashen')
library_info(BOA_APP_ID,'BOA')
library_info(CBE_APP_ID,'CBE')

Dashen Bank App Info
App Title   : Dashen Bank
Current Score: 4.24618
Total Ratings: 5,675
Total Reviews: 1,026
Installs     : 1,000,000+
BOA Bank App Info
App Title   : BoA Mobile
Current Score: 4.3939075
Total Ratings: 9,265
Total Reviews: 1,463
Installs     : 1,000,000+
CBE Bank App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.287285
Total Ratings: 48,500
Total Reviews: 9,323
Installs     : 5,000,000+


In [4]:
def scrape_reviews(BANK_APP_ID,name):
    # Step 2: Scrape reviews
    print(f"Scraping reviews for {name} Bank...")

    result, continuation_token = reviews(
        BANK_APP_ID,
        lang='en',
        country='et',
        sort=Sort.NEWEST,       # Most recent first
        count=500,              # Ask for more than 400 to be safe
        filter_score_with=None  # All star ratings
    )

    print(f"Collected {len(result)} raw reviews")
    return result
    
dashen_result=scrape_reviews(DASHEN_APP_ID,'Dashen')
boa_result=scrape_reviews(BOA_APP_ID,'BOA')
cbe_result=scrape_reviews(CBE_APP_ID,'CBE')

Scraping reviews for Dashen Bank...
Collected 500 raw reviews
Scraping reviews for BOA Bank...
Collected 500 raw reviews
Scraping reviews for CBE Bank...
Collected 500 raw reviews


In [5]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(cbe_result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in cbe_result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: f11ba9ef-c1a1-4006-9ead-afb585624f63
  userName: Jalane Wakweya
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjW6qzoLoNMhou8o4YjjzLMAu8DWOjfERRON3DkfpiofnqaQT4i0rg
  content: Good
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 5.3.0
  at: 2026-05-16 19:03:11
  replyContent: None
  repliedAt: None
  appVersion: 5.3.0


In [6]:
def extract_columns(result,bank_name):
    # Step 3: Extract only the columns we need
    raw_data = []

    for r in result:
        raw_data.append({
            'review_id': r.get('reviewId', ''),
            'review'   : r.get('content', ''),
            'rating'   : r.get('score', None),
            'date'     : r.get('at', None),
            'bank'     : f'{bank_name} Bank',
            'source'   : 'Google Play'
        })

    # Build a DataFrame
    df_raw = pd.DataFrame(raw_data)

    print(f"Shape: {df_raw.shape}")
    df_raw.head()
    return df_raw
    
df_dashen_raw=extract_columns(dashen_result,'Dashen')
df_boa_raw=extract_columns(boa_result,'BOA')
df_cbe_raw=extract_columns(cbe_result,'CBE')

Shape: (500, 6)
Shape: (500, 6)
Shape: (500, 6)


In [7]:
# Basic shape and types
print(f"Total reviews collected: {len(df_cbe_raw)}")
print(f"\nColumn dtypes:")
print(df_cbe_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [8]:
def rating_distribution_and_date(df_raw):
    # Rating distribution — what do users think?
    print("Rating distribution:")
    rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
    for rating, count in rating_counts.items():
        bar = '█' * (count // 5)
        print(f"  {int(rating)} stars: {count:>4}  {bar}")
    
    # What does the date column look like right now?
    print("Sample date values (raw):")
    print(df_raw['date'].head(10).to_string())

    print(f"\nDate dtype: {df_raw['date'].dtype}")
        
rating_distribution_and_date(df_cbe_raw)

Rating distribution:
  5 stars:  337  ███████████████████████████████████████████████████████████████████
  4 stars:   45  █████████
  3 stars:   33  ██████
  2 stars:   12  ██
  1 stars:   73  ██████████████
Sample date values (raw):
0   2026-05-16 19:03:11
1   2026-05-16 15:50:50
2   2026-05-16 12:15:55
3   2026-05-16 09:17:00
4   2026-05-16 07:18:33
5   2026-05-16 03:43:47
6   2026-05-15 23:20:32
7   2026-05-15 20:11:22
8   2026-05-15 19:53:26
9   2026-05-15 12:22:49

Date dtype: datetime64[us]


In [9]:
def check_missing_values(df_raw):
    print("=" * 50)
    print("DATA QUALITY AUDIT")
    print("=" * 50)

    # --- Problem 1: Missing Values ---
    print("\nProblem 1: Missing Values")
    print("-" * 30)
    missing = df_raw.isnull().sum()
    missing_pct = (missing / len(df_raw) * 100).round(2)

    for col in df_raw.columns:
        status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
        print(f"  {col:<15}: {status}")

check_missing_values(df_dashen_raw)
# check_missing_values(df_boa_raw)
# check_missing_values(df_cbe_raw)

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


There are no missing values in all of the scraped data of the three banks CBE,DASHEN,BOA.

In [10]:
def check_duplicate(df_raw,name):    
    # --- Problem 2: Duplicate Reviews ---
    print(f"{name} Duplicates")
    print("-" * 30)

    # Exact duplicates on review text
    exact_dupes = df_raw.duplicated(subset=['review']).sum()
    print(f"  Exact duplicate reviews : {exact_dupes}")

    # Duplicate review IDs
    id_dupes = df_raw.duplicated(subset=['review_id']).sum()
    print(f"  Duplicate review IDs    : {id_dupes}")

    # Empty reviews (also a form of bad data)
    empty_reviews = (df_raw['review'].str.strip() == '').sum()
    print(f"  Empty review texts      : {empty_reviews}")
    
check_duplicate(df_dashen_raw,'Dashen')
check_duplicate(df_boa_raw,'BOA')
check_duplicate(df_cbe_raw,'CBE')

Dashen Duplicates
------------------------------
  Exact duplicate reviews : 77
  Duplicate review IDs    : 0
  Empty review texts      : 0
BOA Duplicates
------------------------------
  Exact duplicate reviews : 90
  Duplicate review IDs    : 0
  Empty review texts      : 0
CBE Duplicates
------------------------------
  Exact duplicate reviews : 125
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [11]:
def check_date_format(df_raw):
    # --- Problem 3: Date Format ---
    print("Problem 3: Date Format")
    print("-" * 30)
    print(f"  Current dtype: {df_raw['date'].dtype}")
    print(f"  Sample values: {df_raw['date'].iloc[0]}")
    print(f"  Target format: YYYY-MM-DD (string or date object)")

check_date_format(df_dashen_raw)
# check_date_format(df_boa_raw)
# check_date_format(df_cbe_raw)

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-16 20:55:36
  Target format: YYYY-MM-DD (string or date object)


DATA CLEANING

In [12]:
# Work on a copy so raw data stays untouched
df_dashen = df_dashen_raw.copy()
df_boa=df_boa_raw.copy()
df_cbe=df_cbe_raw.copy()

print(f"Starting with: {len(df_dashen)} reviews")

Starting with: 500 reviews


In [13]:
def handle_missingValues(df):
    before = len(df)

    # Drop rows missing the critical columns
    critical_cols = ['review', 'rating']
    df = df.dropna(subset=critical_cols)

    removed = before - len(df)
    print(f"Removed {removed} rows with missing critical data")
    print(f"Remaining: {len(df)} reviews")
    
handle_missingValues(df_dashen)
# handle_missingValues(df_boa)
# handle_missingValues(df_cbe)

Removed 0 rows with missing critical data
Remaining: 500 reviews


In [14]:
def remove_duplicates(df):
    before = len(df)

    df = df.drop_duplicates(subset=['review_id'], keep='first')

    removed = before - len(df)
    print(f"Removed {removed} duplicate reviews")
    print(f"Remaining: {len(df)} reviews")
    
remove_duplicates(df_dashen)
remove_duplicates(df_boa)
remove_duplicates(df_cbe)

Removed 0 duplicate reviews
Remaining: 500 reviews
Removed 0 duplicate reviews
Remaining: 500 reviews
Removed 0 duplicate reviews
Remaining: 500 reviews


In [15]:
def normalize_date(df):
    print("Before normalization:")
    print(df['date'].head(3).to_string())
    print(f"dtype: {df['date'].dtype}")

    # Convert to pandas datetime, then format as YYYY-MM-DD string
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

    print("\nAfter normalization:")
    print(df['date'].head(3).to_string())
    print(f"dtype: {df['date'].dtype}")

    print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
    
# normalize_date(df_dashen)
# normalize_date(df_boa)
normalize_date(df_cbe)

Before normalization:
0   2026-05-16 19:03:11
1   2026-05-16 15:50:50
2   2026-05-16 12:15:55
dtype: datetime64[us]

After normalization:
0    2026-05-16
1    2026-05-16
2    2026-05-16
dtype: str

Date range: 2026-03-04 to 2026-05-16


CLEAN REVIEW TEXT

In [16]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text


# Apply to the full column
df_dashen['review'] = df_dashen['review'].apply(clean_text)
df_boa['review'] = df_boa['review'].apply(clean_text)
df_cbe['review'] = df_cbe['review'].apply(clean_text)

def remove_empty_reviews(df):
    # Remove any reviews that became empty after cleaning
    before = len(df)
    df = df[df['review'].str.len() > 0]
    removed = before - len(df)
    print(f"\nRemoved {removed} reviews that were empty after cleaning")

remove_empty_reviews(df_dashen)
remove_empty_reviews(df_boa)
remove_empty_reviews(df_cbe)



Removed 0 reviews that were empty after cleaning

Removed 0 reviews that were empty after cleaning

Removed 0 reviews that were empty after cleaning


VALIDATE RATINGS

In [17]:
def validate_ratings(df):
    # Check for out-of-range ratings
    invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
    print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

    # Remove them
    df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

    # Ensure rating is stored as integer
    df['rating'] = df['rating'].astype(int)

    print(f"Remaining: {len(df)} reviews")
    print(f"Rating dtype: {df['rating'].dtype}")

validate_ratings(df_dashen)
validate_ratings(df_boa)
validate_ratings(df_cbe)

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64
Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64
Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


In [18]:
def produce_clean_dateset(df):
    # Select only the 5 required columns in the right order
    df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

    # Sort by date (newest first) for clean presentation
    df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

    print(f"Final dataset shape: {df_clean.shape}")
    # df_clean.head(10)
    return df_clean

df_dashen_clean=produce_clean_dateset(df_dashen)
df_boa_clean=produce_clean_dateset(df_boa)
df_cbe_clean=produce_clean_dateset(df_cbe)

df_dashen_clean.head()

Final dataset shape: (500, 5)
Final dataset shape: (500, 5)
Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,Misguiding - Claimed anyone can convert ETB to...,1,2026-05-16 20:55:36,Dashen Bank,Google Play
1,good,5,2026-05-16 17:57:10,Dashen Bank,Google Play
2,ok,5,2026-05-15 20:41:19,Dashen Bank,Google Play
3,best app so far. thank you,5,2026-05-15 18:33:32,Dashen Bank,Google Play
4,Very Annoying App i tried to open virtual bank...,1,2026-05-14 19:33:42,Dashen Bank,Google Play


In [22]:
# Save to CSV
output_path = '../data/processed/dshen_bank_reviews_clean.csv'
df_dashen_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/dshen_bank_reviews_clean.csv


In [24]:
def preprocessing_report(df_raw,df_clean,name):    
    print("=" * 55)
    print(f"  PREPROCESSING REPORT — {name} Bank Reviews")
    print("=" * 55)

    original_count = len(df_raw)
    final_count    = len(df_clean)
    removed_total  = original_count - final_count
    retention_rate = (final_count / original_count * 100)

    print(f"\n  Raw reviews collected  : {original_count:>6}")
    print(f"  Reviews after cleaning : {final_count:>6}")
    print(f"  Reviews removed        : {removed_total:>6}")
    print(f"  Data retention rate    : {retention_rate:>5.1f}%")

    quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
    print(f"  Data quality           : {quality}")

    print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

    print("\n  Rating distribution:")
    for rating in sorted(df_clean['rating'].unique(), reverse=True):
        count = (df_clean['rating'] == rating).sum()
        pct   = count / final_count * 100
        bar   = '█' * (count // 5)
        print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

    print("\n  Text length stats:")
    lengths = df_clean['review'].str.len()
    print(f"    Min    : {lengths.min()} characters")
    print(f"    Median : {lengths.median():.0f} characters")
    print(f"    Max    : {lengths.max()} characters")

    print("\n  Columns in final CSV:")
    for col in df_clean.columns:
        print(f"    - {col}")

    print("\n" + "=" * 55)
    
preprocessing_report(df_dashen_raw,df_dashen_clean,'Dashen')
preprocessing_report(df_boa_raw,df_boa_clean,'BOA')
preprocessing_report(df_cbe_raw,df_cbe_clean,'CBE')

  PREPROCESSING REPORT — Dashen Bank Reviews

  Raw reviews collected  :    500
  Reviews after cleaning :    500
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-08-14 15:16:24  to  2026-05-16 20:55:36

  Rating distribution:
    5 stars :  323 (64.6%)  ████████████████████████████████████████████████████████████████
    4 stars :   33 ( 6.6%)  ██████
    3 stars :   27 ( 5.4%)  █████
    2 stars :   19 ( 3.8%)  ███
    1 stars :   98 (19.6%)  ███████████████████

  Text length stats:
    Min    : 1 characters
    Median : 22 characters
    Max    : 499 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

  PREPROCESSING REPORT — BOA Bank Reviews

  Raw reviews collected  :    500
  Reviews after cleaning :    500
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-02-23 11:06:47  to  2026-05